In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import tensorflow as tf
#from tensorflow.keras import layers, models, losses
#from tensorflow.keras.callbacks import ModelCheckpoint
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

I0000 00:00:1780258786.527813    4746 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780258786.755931    4746 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780258788.366283    4746 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# [LOG] Model Versioning

## Version 1: Tiny-Baseline (Obsolete)
**Data:** 15/05/2026
**Fase:** Upper-Bound Baseline (Test di fattibilità hardware)

### Architettura:
- **Input:** (1, 120, 18) -> [H, W, Channels]
- **Feature Extraction:** - Conv2D (16 filtri, kernel 1x5) + MaxPooling (1x2)
    - SeparableConv2D (32 filtri, kernel 1x3) + MaxPooling (1x2)
- **Output Heads:** - `coords_head`: Dense(8) [Linear] -> X, Y per 4 persone.
    - `mask_head`: Dense(4) [Sigmoid] -> Presenza per 4 persone.

### Statistiche:
- **Parametri Totali:** ~3,500
- **Peso Modello (Float32):** 13.67 KB
- **Peso Stimato (INT8 Quantized):** ~3.5 KB
- **Performance (Epoca 15):** - `val_loss`: 3.79
    - `val_coords_loss`: 3.35 (Errore spaziale medio ~1.83m)
    - `val_mask_loss`: 0.88

## Version 2: Capacità Espansa (Obsolete)
* **Performance:** Errore medio ~1.69m. 
* **Note:** La rete ha smesso di imparare dopo 37 epoche. Mancanza di regolarizzazione (Dropout) e LR fisso.

## Version 3: Architettura "Romana" (Heavy + Callbacks)
**Data:** 31/05/2026
**Fase:** Ottimizzazione Avanzata
* **Architettura:** 12 Layer. Doppie Conv2D(32) -> Doppie SepConv2D(64) -> Conv2D(128) -> Dense(128) + Dropout(0.3) -> Dense(64).
* **Data Pipeline:** `alpha=0.20` (EMA decluttering veloce, come da specifiche). Split dataset con casi complessi (3/4 persone) nel Training. `Batch_Size=8`.
* **Training Hacks:** - `ReduceLROnPlateau`: dimezza il learning rate se la loss si blocca.
    - `EarlyStopping`: ferma l'addestramento se non migliora per 10 epoche e ricarica i pesi migliori.
    - Metrica `RootMeanSquaredError`: legge l'errore spaziale direttamente in Metri.
* **Performance Spaziale:** L'errore medio sulle coordinate è sceso al minimo storico di **~1.56m**.

## Version 4: Architettura "Imperiale" (Obsolete - Disastroso / Mode Collapse)
**Data:** 31/05/2026
**Fase:** Ottimizzazione Matematica e Fallimento Spaziale

### Modifiche Apportate (La Teoria):
- **Spatial Sorting:** Ordinamento forzato dei target da sinistra a destra sull'asse X nel DataGenerator per fornire una regola fissa alla rete (aggirare la *permutation invariance*).
- **True Masked MSE:** Modifica della Loss function per moltiplicare l'errore per 0 quando la persona non è presente, smettendo di penalizzare la rete per i "fantasmi".
- **Bilanciamento Loss:** Il peso della `mask_head` è stato portato da 0.5 a 5.0 per costringere l'ottimizzatore a prestare attenzione alla presenza.
- **Custom Metric:** Creata `true_masked_rmse_metres` per calcolare correttamente l'errore in metri gestendo gli array concatenati a 12 valori (8 coords + 4 maschere).

### Statistiche e Limiti Hardware:
- Aumentare indiscriminatamente i filtri a 128 e i layer Densi a 256 ha causato un **esplosione della memoria Flash stimata a 927.42 KB**, superando il limite tassativo di 800 KB dell'ESP32. 

### Il Disastro (Performance):
- Rispetto al modello V3 (e allo split di Davide), **la resa visiva è pessima**. 
- **Sintomo:** Nel visualizzatore, le predizioni (le X rosse) non inseguono minimamente i bersagli. Rimangono immobili, raggruppate e appiccicate in un singolo punto nell'angolo in basso a sinistra della stanza.
- **Causa:** Il layer `GlobalAveragePooling2D`. Calcolando la media matematica su tutta l'ultima mappa di estrazione, ha letteralmente distrutto ogni informazione geometrica. La rete è diventata **cieca**. Non sapendo *dove* guardare, ha applicato un "Mode Collapse": ha imparato a sparare tutte le previsioni nel punto medio statistico per subire la minor penalità possibile dalla Loss.

----

# [GUIDA] Gerarchia del Fine-Tuning per Edge AI (ESP32-S3)

Nel TinyML non possiamo ingrandire la rete a caso, perché siamo limitati da 400KB di RAM e dalla latenza. Le modifiche seguono un ordine di priorità basato sul **Costo Hardware**.

### Livello 1: Costo Hardware ZERO (Modifiche di Addestramento)
Questi parametri non alterano il peso finale del file `.tflite`. Si provano per primi.
* **1. Epoche (`epochs`):** * *Cos'è:* Il tempo di studio. Quante volte la rete vede l'intero dataset.
    * *Quando usarlo:* Se la `val_loss` sta scendendo ma l'addestramento finisce troppo presto (Underfitting).
    * *Effetto:* Permette alla rete di continuare a correggere gli errori.
* **2. Learning Rate (`lr`):**
    * *Cos'è:* La "lunghezza del passo" durante la discesa del gradiente.
    * *Quando usarlo:* Se la Loss salta su e giù in modo impazzito (LR troppo alto) o se non scende per niente fin dall'inizio (LR troppo basso).
    * *Effetto:* Rende l'apprendimento più stabile o più aggressivo.

### Livello 2: Costo Hardware BASSO (Capacità / Larghezza)
* **3. Numero di Filtri (es. da 32 a 64):**
    * *Quando usarlo:* Se la rete è troppo "stupida" per capire le dinamiche della stanza e la Loss si blocca su valori alti (come nella nostra V1).
    * *Effetto:* Aumenta i parametri (Flash) e leggermente la RAM. Dà alla rete più "neuroni" per capire la trigonometria.

### Livello 3: Costo Hardware ALTO (Profondità / Latenza)
* **4. Aggiungere Layer (es. una terza Conv2D):**
    * *Cos'è:* Aggiungere step sequenziali al modello.
    * *Quando usarlo:* Solo se la rete larga non basta per estrarre concetti complessi.
    * *Effetto:* Aumenta drasticamente le operazioni matematiche (MACs). **Aumenta la latenza:** l'ESP32 ci metterà molto più tempo a calcolare ogni singolo frame. Usare con estrema cautela.

In [9]:
# ==============================================================================
# IL GENERATORE DI DATI (DATA ENGINE) - V4 (Con Spatial Sorting)
# ==============================================================================
class EEAIDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, batch_size=8, alpha=0.20, is_training=True):
        self.file_paths = file_paths
        self.batch_size = batch_size
        self.alpha = alpha
        self.is_training = is_training
        if self.is_training:
            np.random.shuffle(self.file_paths)

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_files = self.file_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch, y_coords_mask_batch, y_mask_batch = [], [], []

        for file_path in batch_files:
            data = np.load(file_path)
            raw_iq = data['radar_cir_iq']   
            people_xy = data['people_xy']   
            people_mask = data['people_mask'] 
            T = raw_iq.shape[0]             
            
            mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
            mag_reshaped = mag.reshape(T, 1, 120, 18) 
            
            bg = np.copy(mag_reshaped[0])
            decluttered = np.zeros_like(mag_reshaped)
            
            # Inizializziamo array vuoti per i target ordinati
            sorted_coords = np.zeros((T, 4, 2), dtype=np.float32)
            
            for t in range(T):
                # 1. EMA Decluttering
                bg = self.alpha * mag_reshaped[t] + (1 - self.alpha) * bg
                decluttered[t] = np.abs(mag_reshaped[t] - bg)
                
                # 2. SPATIAL SORTING
                valid_idx = people_mask[t] > 0.5
                coords = people_xy[t][valid_idx] # Prende solo chi c'è davvero
                
                if len(coords) > 0:
                    # Ordina da sinistra a destra sull'asse X
                    sort_order = np.argsort(coords[:, 0])
                    sorted_valid = coords[sort_order]
                    sorted_coords[t, :len(sorted_valid)] = sorted_valid
                    
            # Appiattiamo a (T, 8)
            flat_coords = sorted_coords.reshape(T, 8)
            
            # Uniamo coordinate e maschere -> shape (T, 12) per la True Masked MSE
            combined_target = np.concatenate([flat_coords, people_mask], axis=1)

            X_batch.append(decluttered)
            y_coords_mask_batch.append(combined_target)
            y_mask_batch.append(people_mask)

        X = np.concatenate(X_batch, axis=0).astype(np.float32)       
        Y_combined = np.concatenate(y_coords_mask_batch, axis=0).astype(np.float32) 
        Y_mask = np.concatenate(y_mask_batch, axis=0).astype(np.float32)   
        
        return X, {"coords_head": Y_combined, "mask_head": Y_mask}

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.file_paths)

# --- INIZIALIZZAZIONE CON IL NUOVO SPLIT DI DAVIDE E ALFA CORRETTO A 0.20 ---
train_indices = [22, 0, 1, 2, 3, 10, 14, 18, 19, 21, 8, 9, 12, 5, 6, 4, 7, 13]
val_indices = [23, 16, 11, 17, 15, 20]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

BATCH_SIZE = 8 
# ALFA IMPOSTATO A 0.20!
train_gen = EEAIDataGenerator(train_files, batch_size=BATCH_SIZE, alpha=0.20, is_training=True)
val_gen = EEAIDataGenerator(val_files, batch_size=BATCH_SIZE, alpha=0.20, is_training=False)

print(f"Motore V4 pronto: {len(train_files)} file di Train, {len(val_files)} file di Validation.")

Motore V4 pronto: 18 file di Train, 6 file di Validation.


In [10]:
def true_masked_mse(y_true_combined, y_pred_coords):
    """
    Riceve 12 valori veri (8 coords + 4 maschere) e 8 predizioni.
    Annulla l'errore per le persone non fisicamente presenti.
    """
    y_true_coords = y_true_combined[:, :8]
    mask_1d = y_true_combined[:, 8:] # shape (batch, 4)
    
    # Duplica la maschera per coprire X e Y: [m1, m1, m2, m2, m3, m3, m4, m4]
    mask_2d = tf.repeat(mask_1d, 2, axis=1) 
    
    # Errore quadratico standard
    raw_mse = tf.square(y_true_coords - y_pred_coords)
    
    # Mascheratura: moltiplica l'errore per 0 se la persona non c'è
    masked_mse = raw_mse * mask_2d
    
    # Media solo sugli elementi validi per non falsare l'errore
    valid_elements = tf.reduce_sum(mask_2d) + 1e-6
    return tf.reduce_sum(masked_mse) / valid_elements

In [11]:
def embedded_summary(model, input_shape=(1, 120, 18)):
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
    max_layer_ram_kb = 0
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    print("============================================")
    print("   REPORT REQUISITI ESP32-S3 (FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite: 400 KB)")
    print("============================================\n")

In [12]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V4 "Imperiale" (Ottimizzata per 300KB SRAM)
# ==============================================================================
def build_eeai_model_v4_imperiale(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # Blocco 1: Base Allargata
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x)
    x = layers.Conv2D(64, (1, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) 
    
    # Blocco 2: Deep Spatial Features
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x)
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_4")(x) 
    
    # Blocco 3: Rifinitura
    x = layers.Conv2D(256, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    # Collo di Bottiglia Espanso + Dropout Regolarizzatore
    x = layers.Dense(256, activation='relu', name="features_deep")(x)
    x = layers.Dropout(0.4, name="drop_features")(x) 
    common_feat = layers.Dense(128, activation='relu', name="features")(x)

    # Output Heads
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_V4_Imperiale")

model_v4 = build_eeai_model_v4_imperiale()

# Stampiamo il check hardware!
embedded_summary(model_v4)

# --- IL FIX: METRICA CUSTOM PER L'RMSE ---
def true_masked_rmse_metres(y_true_combined, y_pred_coords):
    """
    Calcola la radice quadrata dell'errore (RMSE) ignorando i fantasmi.
    Questo stamperà a schermo l'errore REALE in metri durante il training!
    """
    y_true_coords = y_true_combined[:, :8]
    mask_1d = y_true_combined[:, 8:] 
    mask_2d = tf.repeat(mask_1d, 2, axis=1) 
    
    raw_mse = tf.square(y_true_coords - y_pred_coords)
    masked_mse = raw_mse * mask_2d
    
    valid_elements = tf.reduce_sum(mask_2d) + 1e-6
    mean_mse = tf.reduce_sum(masked_mse) / valid_elements
    return tf.sqrt(mean_mse)

# COMPILAZIONE
model_v4.compile(
    optimizer='adam',
    loss={"coords_head": true_masked_mse, "mask_head": "binary_crossentropy"}, 
    
    # IL MEGAFONO: Maschera pesa 5.0
    loss_weights={"coords_head": 1.0, "mask_head": 5.0},
    
    metrics={
        # Inseriamo la nostra nuova metrica custom che sa leggere i 12 valori
        "coords_head": [true_masked_rmse_metres],
        "mask_head": [tf.keras.metrics.BinaryAccuracy(name="bin_acc")]
    }
)

# --- CALLBACKS INTELLIGENTI ---
checkpoint_v4 = ModelCheckpoint("eeai_best_model_v4_imperiale.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI 
print("\n--- INIZIO ADDESTRAMENTO V4 IMPERIALE ---")
history_v4 = model_v4.fit(
    train_gen,                
    validation_data=val_gen,  
    epochs=100,
    callbacks=[checkpoint_v4, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

   REPORT REQUISITI ESP32-S3 (FLOAT32)   
 Memoria FLASH stimata : 927.42 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~8.44 KB (Limite: 400 KB)


--- INIZIO ADDESTRAMENTO V4 IMPERIALE ---
Epoch 1/100
1/3 ━━━━━━━━━━━━━━━━━━━━ 9s 5s/step - coords_head_loss: 11.8607 - coords_head_true_masked_rmse_metres: 3.4439 - loss: 15.3183 - mask_head_bin_acc: 0.5065 - mask_head_loss: 0.6915

KeyboardInterrupt: 

### Come leggere le Loss del nostro Multi-Head Model

1. **coords_head_loss (L'Errore di Posizione - MSE)**
Misura la distanza matematica al quadrato. Esempio: se vale `3.35`, l'errore medio in metri è la radice quadrata (circa `1.83` metri). Più scende, più le X rosse si avvicinano ai pallini verdi.

2. **mask_head_loss (L'Errore di Presenza - Binary Crossentropy)**
Misura la confusione della rete sulla presenza o meno della persona (0 o 1). Più è bassa, meno fantasmi vedremo.

3. **loss (La Loss Totale)**
Il voto complessivo: `coords_loss * 1.0 + mask_loss * 5`. L'ottimizzatore cerca di abbassare questo numero il più possibile.

4. **val_loss (validation loss)**
È l'errore che la rete commette sui dati che non ha mai visto (l'esame di fine modulo).


In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.0 (Anti-Sfarfallio e Ground Truth Fixata)
# ==============================================================================

file_target = "dataset/data/window_000015.npz"
#file_target = "dataset/data/window_000011.npz"
#file_target = "dataset/window_000015.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V3)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    # Usa esplicitamente il modello appena addestrato!
    preds = model_v4.predict(decluttered, verbose=0)
    p_coords = preds[0].reshape(T, 4, 2)
    p_mask = preds[1]
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_title(f"Radar V4 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V3)...
Dati pronti! Inizializzazione Radar...
